# Full-video diarization test
Private GPU test of the tagged two-minute milestone, with GPU-enabled SpeechBrain and InsightFace and completed-stage checkpoints. Attribution rules are unchanged.

1. Create a **private Kaggle Dataset** by uploading `kaggle_diarization_bundle.zip`; let Kaggle extract it.
2. Import this notebook into Kaggle; keep the notebook **private**, attach that dataset, turn **Internet on**, and select an NVIDIA **GPU** accelerator.
3. In **Add-ons → Secrets**, add `HF_TOKEN` and grant this notebook access. The associated Hugging Face account must already have accepted access terms for the diarization models.
4. Run cells in order. The full-video cell is separate from the two-minute compatibility test.

No external Git repository or public upload is required. Cloud GPU execution has not yet been tested. GPU arithmetic and fresh ASR can vary from the saved CPU result.

In [ ]:
from pathlib import Path
import subprocess, sys, shutil, os, json, time, zipfile
candidates = list(Path('/kaggle/input').rglob('chainofrules.py'))
if len(candidates) != 1:
    raise RuntimeError('Attach only the private diarization dataset; expected exactly one chainofrules.py.')
DATA = candidates[0].parent
WORK = Path('/kaggle/working/diarization')
WORK.mkdir(parents=True, exist_ok=True)
for name in ('chainofrules.py', 'cloud_runtime.py', 'requirements-kaggle.txt',
             'test_cloud_runtime.py', 'test_short_answers.py'):
    shutil.copy2(DATA/name, WORK/name)
VENV = Path('/kaggle/working/diarization-venv')
PYTHON = str(VENV/'bin/python')
# A failed stdlib venv can leave bin/python present but pip missing.
ready = False
if Path(PYTHON).exists():
    probe = subprocess.run([PYTHON, '-m', 'pip', '--version'],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    ready = probe.returncode == 0
if not ready:
    # virtualenv seeds pip from bundled wheels instead of relying on ensurepip.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--user', 'virtualenv>=20.26,<21'], check=True)
    subprocess.run([sys.executable, '-m', 'virtualenv', '--no-download', str(VENV)], check=True)
subprocess.run([PYTHON, '-m', 'pip', '--version'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
# Isolated environment: do not replace Kaggle's notebook dependencies.
subprocess.run([PYTHON, '-m', 'pip', 'install', '-r', str(WORK/'requirements-kaggle.txt')], check=True)
if shutil.which('ffmpeg') is None:
    raise RuntimeError('This runner requires ffmpeg in the Kaggle image.')
subprocess.run([PYTHON, '-m', 'pip', 'check'], check=True)
# InsightFace can pull the CPU distribution, which shares files with the GPU one.
# Run after installing all dependencies. Keep exactly one ONNX Runtime distribution.
subprocess.run([PYTHON, '-m', 'pip', 'uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', '--no-deps', '--force-reinstall',
                'onnxruntime-gpu==1.23.2'], check=True)
subprocess.run([PYTHON, '-c',
    "import onnxruntime as ort; print('ONNX Runtime:', ort.__version__, ort.__file__); print('Providers:', ort.get_available_providers()); assert 'CUDAExecutionProvider' in ort.get_available_providers(), 'CUDA provider missing from installed ONNX Runtime'"], check=True)


## GPU and secret setup
This uses one GPU. Face sessions list their **actual** providers when the test starts. If they show only CPU, inspect that warning before committing to the full run.

In [ ]:
from kaggle_secrets import UserSecretsClient
ENV = os.environ.copy()
# The child venv runs without the notebook inline-display backend.
ENV['MPLBACKEND'] = 'Agg'
ENV['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
ENV['PYTHONUNBUFFERED'] = '1'
ENV['PYTHONDONTWRITEBYTECODE'] = '1'
ENV['MPLCONFIGDIR'] = '/kaggle/working/matplotlib'
# Make pip-installed CUDA libraries visible to CTranslate2 in the child process.
library_dirs = subprocess.check_output([PYTHON, '-c',
    "import site,pathlib; print(':'.join(str(p) for d in site.getsitepackages() for p in pathlib.Path(d).glob('nvidia/*/lib')))"], text=True).strip()
ENV['LD_LIBRARY_PATH'] = library_dirs + ':' + ENV.get('LD_LIBRARY_PATH', '')
subprocess.run([PYTHON, '-c',
    "import torch,onnxruntime as ort; ort.preload_dlls(); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'); print('ORT:',ort.get_available_providers()); assert torch.cuda.is_available(), 'Select a GPU accelerator'; assert 'CUDAExecutionProvider' in ort.get_available_providers(), 'GPU ONNX Runtime is missing'"], env=ENV, check=True)
subprocess.run([PYTHON, '-m', 'unittest', 'test_cloud_runtime', 'test_short_answers'], cwd=WORK, env=ENV, check=True)
RESULTS = Path('/kaggle/working/results')
RESULTS.mkdir(exist_ok=True)
CACHE = Path('/kaggle/working/stage-cache')
# To resume a later session, attach a private dataset containing the downloaded
# stage-checkpoints.zip. Exact matching stages are reused; weights redownload.
archives = list(Path('/kaggle/input').rglob('stage-checkpoints.zip'))
for archive in archives:
    with zipfile.ZipFile(archive) as zipped:
        # Only restore checkpoint files beneath the working directory.
        for member in zipped.infolist():
            target = (Path('/kaggle/working')/member.filename).resolve()
            if not target.is_relative_to(Path('/kaggle/working/stage-cache')):
                raise RuntimeError('Unexpected path in checkpoint archive')
        zipped.extractall('/kaggle/working')

In [ ]:
def export_checkpoints():
    if CACHE.exists():
        shutil.make_archive('/kaggle/working/stage-checkpoints', 'zip', CACHE.parent, CACHE.name)

def run_test(video_name, stem, batch_size=4):
    output = RESULTS/(stem+'_evidence.json')
    command = [PYTHON, str(WORK/'chainofrules.py'), str(DATA/video_name),
               '--voice-priors', str(DATA/'voice_embeddings.npy'),
               '--face-priors', str(DATA/'face_embeddings.npy'),
               '--output', str(output), '--cache-dir', str(CACHE),
               '--batch-size', str(batch_size)]
    started = time.monotonic()
    try:
        with (RESULTS/(stem+'.log')).open('w') as log:
            process = subprocess.Popen(command, cwd=WORK, env=ENV, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                # Never save or display the secret even if a dependency prints it.
                line = line.replace(ENV['HF_TOKEN'], '[REDACTED]')
                log.write(line); log.flush()
                print(line, end='')
            if process.wait() != 0:
                raise RuntimeError('Test failed; see the log. For CUDA out-of-memory, retry with batch_size=1.')
    finally:
        export_checkpoints()
    result = json.loads(output.read_text())
    transcript = '\n'.join(f"[{s['start']:.2f}-{s['end']:.2f}] {s['final_speaker']} (strength={s['final_confidence']:.2f}): {s['text']}" for s in result['segments'])
    (RESULTS/(stem+'_transcript.txt')).write_text(transcript+'\n')
    subprocess.run([PYTHON, '-m', 'pip', 'freeze'], stdout=(RESULTS/'cloud-packages.txt').open('w'), check=True)
    print(f'Elapsed: {(time.monotonic()-started)/60:.1f} minutes')
    print('Runtime:', result.get('runtime'))
    return result

## First: two-minute compatibility test
Use batch size 4 initially to leave GPU memory headroom. The tagged local CPU run used batch size 16. Compare the transcript; do not require identical fresh ASR segmentation.

In [ ]:
two_minute = run_test('longer_2min.mp4', 'two_minute')
print((RESULTS/'two_minute_transcript.txt').read_text())
print('Saved milestone transcript for comparison:')
print((DATA/'milestone_transcript.txt').read_text())

## Then: full-video test
Run this cell after checking the two-minute result and actual face providers. This processes the entire video in one diarization pass. Longer-video clustering may differ; short-phrase corrections remain weak and need review.

In [ ]:
full_video = run_test('video.mp4', 'full_video')

## Download results and checkpoints
Download both links before ending the session. Kaggle working files are temporary unless retained through a saved notebook version or downloaded. A later session can resume **completed** stages by attaching the private checkpoint archive; an interrupted stage runs again. Checkpoints contain transcript and voice/face evidence and should remain private.

In [ ]:
from IPython.display import FileLink, display
export_checkpoints()
shutil.make_archive('/kaggle/working/diarization-results', 'zip', RESULTS)
display(FileLink('/kaggle/working/diarization-results.zip'))
display(FileLink('/kaggle/working/stage-checkpoints.zip'))